# Shi–Horvath synthetic-null unsupervised forest

Persist the synthetic generator and seed, then index observed rows only for declared original/OOB relations.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.ensemble import RandomForestClassifier
from forestgap import Proximity, SyntheticNullForest, proximity_clustering

X, labels = make_blobs(n_samples=N_SAMPLES, centers=3, n_features=5, random_state=RANDOM_STATE)
forest = SyntheticNullForest(
    estimator=RandomForestClassifier(
        n_estimators=N_ESTIMATORS, bootstrap=True, random_state=RANDOM_STATE,
    ),
    random_state=RANDOM_STATE,
).fit(X)
original = Proximity(forest, relation="original").attach(X)
oob = Proximity(forest, relation="oob").attach(X)
clusters = proximity_clustering(original, n_clusters=3, random_state=RANDOM_STATE)
native_observed_scores = forest.score_samples(X)
{
    "generator": forest.generator_,
    "synthetic_seed": forest.synthetic_seed_,
    "synthetic_rows": forest.synthetic_X_.shape[0],
    "original_relation_shape": original.get_proximities().shape,
    "oob_relation_shape": oob.get_proximities().shape,
    "native_observed_score_range": [float(native_observed_scores.min()), float(native_observed_scores.max())],
    "cluster_count": int(np.unique(clusters).size),
}

{'generator': 'independent_marginals',
 'synthetic_seed': 0,
 'synthetic_rows': 80,
 'original_relation_shape': (80, 80),
 'oob_relation_shape': (80, 80),
 'native_observed_score_range': [0.55, 1.0],
 'cluster_count': 3}